# NOTEARS experiment analysis

In [17]:
from pathlib import Path
import json
import warnings
import re

import pandas as pd

# Change these paths if the result directories are elsewhere.
INPUT_DIRS = {
    "linear": Path("linear_exist_edge_pairs"),
    "nonlinear": Path("nonlinear_exist_edge_pairs"),
}

# Set this to True only if you want old files whose loss type was not saved.
INCLUDE_LEGACY_UNSPECIFIED = False

ACC_KEYS = {
    "acc_prior_ll": ("with_prior", "likelihood"),
    "acc_prior_l2": ("with_prior", "l2"),
    "acc_no_prior_ll": ("no_prior", "likelihood"),
    "acc_no_prior_l2": ("no_prior", "l2"),
}

if INCLUDE_LEGACY_UNSPECIFIED:
    ACC_KEYS.update({
        "acc_prior": ("with_prior", "unspecified"),
        "acc_no_prior": ("no_prior", "unspecified"),
    })

In [18]:
def load_accuracy_rows(input_dirs):
    rows = []

    for experiment, directory in input_dirs.items():

        for path in sorted(directory.glob("*.json")):
            with path.open(encoding="utf-8") as file:
                result = json.load(file)

            d = len(result.get("B_true", [])) or None
            seed_match = re.search(r"_seed(\d+)\.json$", path.name)
            seed = int(seed_match.group(1)) if seed_match else None
            configuration = re.sub(r"_seed\d+\.json$", "", path.name)
            for key, (prior_setting, loss) in ACC_KEYS.items():
                scores = result.get(key)

                if not isinstance(scores, dict):
                    continue
                
                row = {
                    "experiment": experiment,
                    "prior_setting": prior_setting,
                    "loss": loss,
                    "d": d,
                    "seed": seed,
                    "configuration": configuration,
                    "file": path.name,
                }
                row.update({
                    score_name: score_value
                    for score_name, score_value in scores.items()
                    if isinstance(score_value, (int, float))
                })
                rows.append(row)

    return pd.DataFrame(rows)

accuracy_rows = load_accuracy_rows(INPUT_DIRS)

print(f"Loaded {len(accuracy_rows)} accuracy records from {accuracy_rows['file'].nunique()} JSON files.")
print("The table below contains individual runs; it is not an average.")
accuracy_rows.head().style.set_caption("Raw per-run accuracy records")

Loaded 80 accuracy records from 20 JSON files.
The table below contains individual runs; it is not an average.


,experiment,prior_setting,loss,d,seed,configuration,file,fdr,tpr,fpr,f1,shd,nnz
0,linear,with_prior,likelihood,5,0,linear_exist_edge_pairs_ER1_d5_gauss_rate0.5,linear_exist_edge_pairs_ER1_d5_gauss_rate0.5_seed0.json,0.000000,1.000000,0.000000,1.000000,0,5
1,linear,with_prior,l2,5,0,linear_exist_edge_pairs_ER1_d5_gauss_rate0.5,linear_exist_edge_pairs_ER1_d5_gauss_rate0.5_seed0.json,0.333333,0.800000,0.400000,0.727273,3,6
2,linear,no_prior,likelihood,5,0,linear_exist_edge_pairs_ER1_d5_gauss_rate0.5,linear_exist_edge_pairs_ER1_d5_gauss_rate0.5_seed0.json,0.000000,1.000000,0.000000,1.000000,0,5
3,linear,no_prior,l2,5,0,linear_exist_edge_pairs_ER1_d5_gauss_rate0.5,linear_exist_edge_pairs_ER1_d5_gauss_rate0.5_seed0.json,1.000000,0.000000,0.800000,0.000000,5,4
4,linear,with_prior,likelihood,5,1,linear_exist_edge_pairs_ER1_d5_gauss_rate0.5,linear_exist_edge_pairs_ER1_d5_gauss_rate0.5_seed1.json,0.000000,1.000000,0.000000,1.000000,0,5


In [16]:
metadata_columns = {
    "experiment", "configuration", "prior_setting",
    "loss", "d", "seed", "file",
}
score_columns = [
    column for column in accuracy_rows.columns
    if column not in metadata_columns
    and pd.api.types.is_numeric_dtype(accuracy_rows[column])
]

# Remove only the seed suffix, then average all runs of the same setup.
summary = (
    accuracy_rows
    .groupby(
        ["experiment", "configuration", "d", "prior_setting", "loss"],
        dropna=False,
    )
    .agg(num_seeds=("seed", "nunique"), **{
        score: (score, "mean") for score in score_columns
    })
    .round(4)
)
summary

num_seeds  \
experiment configuration                                 d prior_setting loss                    
linear     linear_exist_edge_pairs_ER1_d5_gauss_rate0.5  5 no_prior      l2                 10   
                                                                         likelihood         10   
                                                           with_prior    l2                 10   
                                                                         likelihood         10   
nonlinear  nonlinear_exist_edge_pairs_ER1_d5_mlp_rate0.5 5 no_prior      l2                 10   
                                                                         likelihood         10   
                                                           with_prior    l2                 10   
                                                                         likelihood         10   

                                                                                        fdr  \
experiment configuration                                 d prior_setting loss                 
linear     linear_exist_edge_pairs_ER1_d5_gauss_rate0.5  5 no_prior      l2          0.5833   
                                                                         likelihood  0.4883   
                                                           with_prior    l2          0.1383   
                                                                         likelihood  0.2367   
nonlinear  nonlinear_exist_edge_pairs_ER1_d5_mlp_rate0.5 5 no_prior      l2          0.5414   
                                                                         likelihood  0.4775   
                                                           with_prior    l2          0.3049   
                                                                         likelihood  0.4399   

                                                                                      tpr  \
experiment configuration                                 d prior_setting loss               
linear     linear_exist_edge_pairs_ER1_d5_gauss_rate0.5  5 no_prior      l2          0.34   
                                                                         likelihood  0.50   
                                                           with_prior    l2          0.78   
                                                                         likelihood  0.74   
nonlinear  nonlinear_exist_edge_pairs_ER1_d5_mlp_rate0.5 5 no_prior      l2          0.48   
                                                                         likelihood  0.82   
                                                           with_prior    l2          0.84   
                                                                         likelihood  0.86   

                                                                                      fpr  \
experiment configuration                                 d prior_setting loss               
linear     linear_exist_edge_pairs_ER1_d5_gauss_rate0.5  5 no_prior      l2          0.48   
                                                                         likelihood  0.46   
                                                           with_prior    l2          0.14   
                                                                         likelihood  0.24   
nonlinear  nonlinear_exist_edge_pairs_ER1_d5_mlp_rate0.5 5 no_prior      l2          0.68   
                                                                         likelihood  0.78   
                                                           with_prior    l2          0.44   
                                                                         likelihood  0.70   

                                                                                         f1  \
experiment configuration                                 d prior_setting loss                 
linear     linear_exist_edge_pairs_ER1_d5_gauss_rate0.5  5 no_prior      l2          0.3722   
      